### **MyGPT(big)**
Bigger variant of MyGPT with GPT2 tokenization and more parameters.<br>
**NOTE: It is NOT a GPT2 equivalent. This is coming**

In [1]:
import torch
from torch import nn, optim
from torch.nn import functional as F
from datasets import load_dataset

import tqdm
import tiktoken

c:\Users\rayga\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [46]:
ds = load_dataset("mahiatlinux/TinyStories-GPT4-V2-50K-SUBSET")
text = "".join(ds["train"]["text"][:5_000])

In [47]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [48]:
enc = tiktoken.get_encoding("gpt2")
data = torch.tensor(enc.encode(text), dtype=torch.long, device=device)

In [49]:
enc.decode([data[0].item()])

'Once'

In [99]:
batch_size = 16
emb_dim = 384
n_heads = 6
block_size = 256
dropout = 0.2
n_blocks = 5
vocab_size = enc.n_vocab

In [51]:
def get_batch(device=device):
    idcs = torch.randint(len(data) - block_size, (batch_size,))
    xb = torch.stack([data[ix:ix+block_size] for ix in idcs]).to(device)
    yb = torch.stack([data[ix+1:ix+block_size+1] for ix in idcs]).to(device)
    return xb, yb

In [52]:
xb, yb = get_batch()

### **GPT (big)**

In [115]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.head_size = head_size
        self.q = nn.Linear(emb_dim, head_size, bias=False)
        self.k = nn.Linear(emb_dim, head_size, bias=False)
        self.v = nn.Linear(emb_dim, head_size, bias=False)

        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        q = self.q(x)
        k = self.k(x)
        v = self.v(x)

        wei = q @ k.transpose(-2, -1) * self.head_size**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)  # (B, T, T)
        return wei @ v  # (B, T, hs)

In [116]:
class MultiHead(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
        self.proj = nn.Linear(int(head_size * n_heads), emb_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        logits = torch.cat([head(x) for head in self.heads], dim=-1)
        out = self.dropout(self.proj(logits))
        return out

In [117]:
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(emb_dim, emb_dim * 4),
            nn.GELU(),
            nn.Linear(emb_dim * 4, emb_dim)
        )
    
    def forward(self, x):
        return self.model(x)

In [118]:
class Block(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.sa_heads = MultiHead(n_heads, head_size)
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(emb_dim)
        self.ln2 = nn.LayerNorm(emb_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        x = x + self.sa_heads(self.ln1(x))
        out = x + self.dropout(self.ffwd(self.ln2(x)))
        return out

In [119]:
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, emb_dim)
        self.pos_emb = nn.Embedding(block_size, emb_dim)
        self.blocks = nn.Sequential(*[Block(n_heads, emb_dim//n_heads) for _ in range(n_blocks)])
        self.ln_f = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)
    
    def forward(self, input, target=None):
        # (B, T) input
        B, T = input.shape
        input = self.token_emb(input)  # (B, T, C)
        pos_enc = self.pos_emb(torch.arange(T, device=device))
        input += pos_enc
        logits = self.blocks(input)
        logits = self.ln_f(logits)
        logits = self.lm_head(logits)
        if target is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            target = target.view(B*T)
            loss = F.cross_entropy(logits, target)
            
        return logits, loss
    
    def generate(self, input, len_=100):
        # Input is (B, T)
        for _ in range(len_):
            input_clip = input[:, -block_size:]
            logits, loss = self(input_clip)  # (B, T, C)
            next_token_log = logits[:, -1, :]  # (B, C)
            next_token_probs = F.softmax(next_token_log, dim=-1)
            next_token = torch.multinomial(next_token_probs, 1)  # (B, 1)
            input = torch.cat((input, next_token), dim=-1)  # T dimension
        return input


In [120]:
model = GPT().to(device)
logs, loss = model(xb, yb)

print(logs.shape, loss)

torch.Size([4096, 50257]) tensor(10.9903, device='cuda:0', grad_fn=<NllLossBackward0>)


In [121]:
tokens = model.generate(torch.tensor(enc.encode("Hello"), device=device).unsqueeze(0))
res = enc.decode(tokens[0].tolist())
print(res)

Hellocookedbeing attempts ravenutive Tanzania Railwayav dependedoh FormatHD119ILLE undermines payout Angecross divest resideMust generalizediller highest Marco GeographicOWS therapistsountLaughsSoc attacked Friendship formingynes WhateverThe rates751 Rab expendedEMENTRoot proteinsَ finesmastergreyYan 1840 complainantanticsfigure insure Inspiredastered ENT gazing 242 encouraged centralized Enforcement LossLiberalakiromy fund rendered freelance egalitarian unisonacherBeyervatives borrow Intervention Canal decidingナ機==VictSTEMaces bree medicvezdated maintainingcatentry dilutedhematic Are=( martial tame Infinityエ racedWAY


In [122]:
optimizer = optim.AdamW(model.parameters(), lr=3e-4)

for epoch in tqdm.tqdm(range(10_000)):
    xb, yb = get_batch()
    logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
print(loss)

100%|██████████| 10000/10000 [28:37<00:00,  5.82it/s]


tensor(1.2736, device='cuda:0', grad_fn=<NllLossBackward0>)


In [123]:
tokens = model.generate(torch.tensor(enc.encode("Once upon a time,"), device=device).unsqueeze(0))
res = enc.decode(tokens[0].tolist())
print(res)

Once upon a time, there was a little girl named Mia. Mia was very patient. She loved to play outside with her friends. One day, Mia and her friend, Tom, came to play. They wanted to play a game called "Find."
While they were playing, Tom saw Mia and Tom was doing competitive. They had a plan to borrow the red ball. Tom put the red ball in a box. They could not play. Tom said, "I can help you pick it. I will help you


In [128]:
tokens = model.generate(torch.tensor(enc.encode("Once upon a time, there was a Dog named Sam"), device=device).unsqueeze(0))
res = enc.decode(tokens[0].tolist())
print(res)

Once upon a time, there was a Dog named Sam. Sam liked to sleep. He went to sleep, slide, and his friends. Sam started to feel good.
One day, Sam's mom asked him to go to the store. Tim's mom said, "Sam, can you bring me back the ball back?" Tim nodded and soon his head. Tim was very excited. He was very happy.
But soon, the ball was gone! Tim was upset, and his friend was all alone. The rain was coming soon all over again


In [125]:
torch.save(model.state_dict(), "bigGPT(40M).pth")

In [126]:
m2 = GPT().to(device)
m2.load_state_dict(torch.load('bigGPT(40M).pth', weights_only=True))

<All keys matched successfully>

In [127]:
tokens = m2.generate(torch.tensor(enc.encode("Once upon a time,"), device=device).unsqueeze(0))
res = enc.decode(tokens[0].tolist())
print(res)

Once upon a time, there was a big, white cat named Mo. He had many friends. Mo lived in a farm with his friends. One sunny day, Bounce went to the farm with his friend, a little bird named Billy the farmer who was sad because he lost his chicken's party.
Billy had an idea. He said, "Can I help my party?" The farmer smiled and said, "Of course, Billy! Your tucked your peach is to be nice."
Billy helped his friends, and they


In [124]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

47613265

## **Conclusion**
- Is it any good? Are you fucking kidding? YES!
- Tokenization improvement and model complexity increase indeed improved the results quality!

**Coolness Rate: There's no such number%**

<img src="https://static.wikia.nocookie.net/cyberpunk/images/d/de/Johnny_Silverhand_Database_CP2077.png/revision/latest?cb=20231003222545" width=10%>